In [1]:
import math
import folium
from weather_data import get_weather

velocity = 1
altitude = 0
density = 1.2
volume = 3
mass = 2.5
gravity = 9.80665               # updated with altitude in the loop
time = 0
dt = 0.1
latitude = 40.8074
longitude = -73.9630
drag_coefficient = 0.8          # subcritical (low Reynolds number) drag coefficient
payload_mass = 2.0
helium_density = 0.166

# Ideal gas model for helium
R_helium = 2077.1          # Specific gas constant for helium (J/kg/K)

#simplified wind constants (fallback if live weather is unavailable)
wind_east = 5.0
wind_north = 2.0

max_altitude = 35000
max_time = 18000

# Live weather (GFS via weather_data.py)
weather_update_interval = 2.0   # simulated seconds between weather lookups
print_interval = 10.0           # simulated seconds between status prints
relative_humidity = 0.0

# Drag crisis: drag coefficient drops once the flow around the balloon turns turbulent
drag_coefficient_supercritical = 0.3
reynolds_crisis_start = 2.0e5
reynolds_crisis_end = 4.0e5

# Added mass: accelerating the balloon also accelerates surrounding air (0.5 for a sphere)
added_mass_coefficient = 0.5

# Burst and parachute descent
burst_enabled = True            # False keeps the old behavior of capping volume at max_volume
max_volume = 100.0              # burst volume (m^3) when burst_enabled is True
parachute_diameter = 1.5        # m
parachute_drag_coefficient = 1.5

# Earth model
earth_radius = 6371000.0        # mean radius for gravity (m)
wgs84_a = 6378137.0             # WGS84 semi-major axis (m)
wgs84_e2 = 6.69437999014e-3     # WGS84 first eccentricity squared


#atmosphere
def atmosphere(altitude):
    altitude = max(0, 0, altitude)
    R = 287.05
    g0 = 9.80665

    layers = [
        (0, 288.15, 101325.0, -0.0065),  # Troposphere,
        (11000, 216.65, 22632.1, 0.0),  # Tropopause
        (20000, 216.65, 5474.89, 0.001),  # Stratosphere
        (32000, 228.65, 868.02, 0.0028),  # Stratopause
        (47000, 270.65, 110.91, 0.0),  # Mesosphere
        (51000, 270.65, 66.94, -0.0028),  # Mesopause
    ]
    if altitude > 51000:
        T = 270.65 - 0.0028 * (altitude - 51000)
        p = 66.94 * math.exp(-g0 * (altitude - 51000) / (R * T))
        return T, p, p / (R * T)
    if altitude > 47000:
        T = 270.65
        p = 110.91 * math.exp(-g0 * (altitude - 47000) / (R * T))
        return T, p, p / (R * T)

    for i in range(len(layers) - 1):
        h0, T0, p0, L = layers[i]
        h1, T1, p1, _ = layers[i + 1]

        if altitude < h1:
            dAltitude = altitude - h0
            if L == 0:
                T = T0
                p = p0 * math.exp(-g0 * dAltitude / (R * T))
            else:
                T = T0 + L * dAltitude
                p = p0 * (T / T0) ** (-g0 / (L * R))
            return T, p, p / (R * T)
    raise ValueError("Atmosphere model altitude out of range.")


#gravity decreases with altitude (inverse square)
def gravity_at(altitude):
    return 9.80665 * (earth_radius / (earth_radius + altitude)) ** 2


#dynamic viscosity of air, Sutherland's law (Pa s)
def air_viscosity(temperature):
    return 1.716e-5 * (temperature / 273.15) ** 1.5 * (273.15 + 110.4) / (temperature + 110.4)


#drag coefficient vs Reynolds number, blended in log(Re) across the drag crisis
def balloon_drag_coefficient(reynolds):
    if reynolds <= reynolds_crisis_start:
        return drag_coefficient
    if reynolds >= reynolds_crisis_end:
        return drag_coefficient_supercritical
    fraction = math.log(reynolds / reynolds_crisis_start) / math.log(reynolds_crisis_end / reynolds_crisis_start)
    return drag_coefficient + fraction * (drag_coefficient_supercritical - drag_coefficient)


#frost point (K): vapor pressure from RH (Bolton, as in weather_data.py), inverted with Magnus over ice
def frost_point(temperature, relative_humidity):
    temp_c = temperature - 273.15
    vapor_pressure = max(relative_humidity, 0.01) / 100.0 * 611.2 * math.exp(17.67 * temp_c / (temp_c + 243.5))
    x = math.log(vapor_pressure / 611.2)
    return 272.62 * x / (22.46 - x) + 273.15


T0, p0, rho0 = atmosphere(0)
helium_mass = helium_density * volume
total_mass = payload_mass + helium_mass

# Load live weather once; fall back to the standard atmosphere and constant wind if it fails
try:
    weather = get_weather(latitude, longitude, altitude)
    wind_east = weather["wind_u_m_s"]
    wind_north = weather["wind_v_m_s"]
    use_live_weather = True
    print(f"Live weather loaded: GFS {weather['forecast_initialization_utc']} +{weather['forecast_hour']}h")
except Exception as error:
    use_live_weather = False
    print(f"Live weather unavailable ({error}); using standard atmosphere and constant wind.")

# Balloon starts out moving with the surface wind
velocity_east = wind_east
velocity_north = wind_north
next_weather_time = 0.0
next_print_time = 0.0
burst = False
burst_point = None

path = [(latitude, longitude)]
records = []
while altitude < max_altitude and time < max_time:
    if use_live_weather and time >= next_weather_time:
        try:
            weather = get_weather(latitude, longitude, altitude)
            temperature = weather["temperature_k"]
            pressure = weather["pressure_pa"]
            density = weather["density_kg_m3"]
            relative_humidity = weather["relative_humidity_percent"]
            wind_east = weather["wind_u_m_s"]
            wind_north = weather["wind_v_m_s"]
        except Exception as error:
            use_live_weather = False
            print(f"Live weather lookup failed ({error}); switching to standard atmosphere.")
        next_weather_time += weather_update_interval
    if not use_live_weather:
        temperature, pressure,density = atmosphere(altitude)
    gravity = gravity_at(altitude)

    if not burst:
        volume = helium_mass * R_helium * temperature / pressure
        if burst_enabled and volume >= max_volume:
            burst = True
            burst_point = (latitude, longitude, altitude, time)
            total_mass = payload_mass   # helium escapes, payload descends under the parachute
            print(f"Balloon burst at {altitude:.1f}m after {time:.1f}s")
        else:
            volume = min(volume, max_volume)

    # Balloon velocity relative to the surrounding air
    relative_east = velocity_east - wind_east
    relative_north = velocity_north - wind_north
    relative_speed = math.sqrt(relative_east**2 + relative_north**2 + velocity**2)

    if burst:
        volume = 0.0
        area = math.pi * (parachute_diameter / 2.0) ** 2
        current_drag_coefficient = parachute_drag_coefficient
    else:
        radius = (3.0 * volume / (4.0 * math.pi)) ** (1.0 / 3.0)
        area = math.pi * radius**2
        reynolds = density * relative_speed * 2.0 * radius / air_viscosity(temperature)
        current_drag_coefficient = balloon_drag_coefficient(reynolds)

    effective_mass = total_mass + added_mass_coefficient * density * volume
    bouyancy = density * volume * gravity
    drag_factor = 0.5 * density * current_drag_coefficient * area * relative_speed
    drag = drag_factor * velocity
    netForce = bouyancy - (total_mass * gravity) - drag
    acceleration = netForce / effective_mass
    velocity += acceleration * dt
    velocity_east += (-drag_factor * relative_east / effective_mass) * dt
    velocity_north += (-drag_factor * relative_north / effective_mass) * dt
    altitude = max(0, altitude + velocity * dt)

    # Position update on the WGS84 ellipsoid
    sin_latitude = math.sin(math.radians(latitude))
    ellipsoid_factor = math.sqrt(1.0 - wgs84_e2 * sin_latitude**2)
    meridian_radius = wgs84_a * (1.0 - wgs84_e2) / ellipsoid_factor**3
    prime_vertical_radius = wgs84_a / ellipsoid_factor
    latitude += math.degrees(velocity_north * dt / (meridian_radius + altitude))
    longitude += math.degrees(velocity_east * dt / ((prime_vertical_radius + altitude) * math.cos(math.radians(latitude))))

    # Icing risk when air below freezing is saturated with respect to ice
    frost_point_k = frost_point(temperature, relative_humidity)
    icing_risk = temperature < 273.15 and temperature <= frost_point_k

    time += dt
    path.append((latitude, longitude))
    records.append({
        "time": time,
        "altitude": altitude,
        "velocity": velocity,
        "acceleration": acceleration,
        "latitude": latitude,
        "longitude": longitude,
        "velocity_east": velocity_east,
        "velocity_north": velocity_north,
        "wind_east": wind_east,
        "wind_north": wind_north,
        "temperature": temperature,
        "pressure": pressure,
        "density": density,
        "relative_humidity": relative_humidity,
        "drag_coefficient": current_drag_coefficient,
        "frost_point": frost_point_k,
        "icing_risk": icing_risk,
        "burst": burst
    })
    if time >= next_print_time:
        print(
            f"Time: {time:.1f}s | "
            f"Altitude: {altitude:.1f}m | "
            f"Velocity: {velocity:.2f}m/s | "
            f"Acceleration: {acceleration}m/s^2 | "
            f"Latitude: {latitude:.5f} | "
            f"Longitude: {longitude:.5f}")
        next_print_time += print_interval

    if burst and altitude <= 0:
        print(f"Landed after {time:.1f}s")
        break

# Create a map centered at the initial position
map_center = path[0]
flight_map = folium.Map(location=map_center, zoom_start=12, tiles=None)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri",
    name="Esri Satellite"
).add_to(flight_map)
folium.PolyLine(path, color="red", weight=4, tooltip="Balloon Trajectory").add_to(flight_map)
folium.Marker(path[0], tooltip="Launch Point", icon=folium.Icon(color="green", icon="play")).add_to(flight_map)
if burst_point is not None:
    folium.Marker(burst_point[:2], tooltip="Burst Point", icon=folium.Icon(color="orange", icon="star"),popup=f"Burst Altitude: {burst_point[2]:.2f}m, \nTime elapsed: {burst_point[3]:.1f}s").add_to(flight_map)
folium.Marker(path[-1], tooltip="Final Point", icon=folium.Icon(color="red", icon="flag"),popup=f"Final Altitude: {altitude:.2f}m, \nTime elapsed: {time:.1f}s").add_to(flight_map)
flight_map.fit_bounds(path)
flight_map.save("balloon_trajectory.html")

print("Map saved as balloon_trajectory.html")

Loading GFS forecast...
✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2026-Oct-26 18:00 UTC F07 ┊ GRIB2 @ aws ┊ IDX @ aws


c:\Users\win3p\CSI\CSI-High-Altitude-Balloons-Ballast\simulation\weather_data.py:97: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  self.fields[name] = xr.concat(
c:\Users\win3p\CSI\CSI-High-Altitude-Balloons-Ballast\simulation\weather_data.py:97: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  self.fields[name] = x

GFS forecast loaded successfully.
Live weather loaded: GFS 2026-10-26T18:00:00Z +7h
Time: 0.1s | Altitude: 0.1m | Velocity: 1.22m/s | Acceleration: 2.211166494133763m/s^2 | Latitude: 40.80740 | Longitude: -73.96299
Time: 10.1s | Altitude: 43.9m | Velocity: 4.86m/s | Acceleration: 0.00029671087309295774m/s^2 | Latitude: 40.80697 | Longitude: -73.96239
Time: 20.0s | Altitude: 92.0m | Velocity: 4.86m/s | Acceleration: 0.00011942573391454811m/s^2 | Latitude: 40.80655 | Longitude: -73.96180
Time: 30.0s | Altitude: 140.7m | Velocity: 4.87m/s | Acceleration: 0.00010720435240877608m/s^2 | Latitude: 40.80613 | Longitude: -73.96120
Time: 40.0s | Altitude: 189.4m | Velocity: 4.87m/s | Acceleration: 0.0001096890965431878m/s^2 | Latitude: 40.80570 | Longitude: -73.96059
Time: 50.0s | Altitude: 238.1m | Velocity: 4.87m/s | Acceleration: 0.0003338669344160563m/s^2 | Latitude: 40.80526 | Longitude: -73.95999
Time: 60.0s | Altitude: 286.9m | Velocity: 4.88m/s | Acceleration: 0.0003379334331630045m/s^2 